# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes 
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [1]:
import plotly.io as pio
pio.renderers.default = 'notebook_connected'   # classic Jupyter / JupyterLab, needs internet

In [2]:
pip install -U plotly nbformat ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


In [5]:
# ---- Shared style (used by every figure) -----------------------------------
# Okabe-Ito colour-blind-safe palette. One category = one colour in every chart.
BLUE, GREY = '#0072B2', '#BBBBBB'          # accent (default) and context grey
CONT_COLORS = {'Africa': '#0072B2', 'Asia': '#E69F00', 'Europe': '#CC79A7',
               'North America': '#D55E00', 'South America': '#56B4E9', 'Oceania': '#009E73'}


def tidy(fig, title, width=1000, height=520, r=40):
    """Decluttered house style: white, left-aligned insight title, no legend, no vertical grid."""
    fig.update_layout(
        template='plotly_white', width=width, height=height,
        title=dict(text=title, x=0, xref='paper', font=dict(family='Arial', size=16)),
        plot_bgcolor='white', paper_bgcolor='white', showlegend=False,
        margin=dict(t=100, l=60, r=r, b=60))
    fig.update_xaxes(showgrid=False, showline=False, tickangle=0, title=None)
    fig.update_yaxes(gridcolor='#EEEEEE', title=None)
    return fig

---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


In [ ]:
Sumerized: 
The question asks how population has changed over time.
A line chart is appropriate because it displays trends and changes across years.
Indexing the population to 1970 = 100 makes continents comparable despite their very different initial population sizes.
The slope of each line shows the relative growth rate.
Highlighting Africa in blue makes the fastest-growing continent immediately visible.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.
in Details:

Chart: Line chart of each continent's population indexed to 1970 = 100. The question is about growth over time *relative to* the 1970 starting size, so slope is the right encoding, and indexing makes continents of very different size comparable (in 1970 Oceania had 19 M people and Asia 2.1 B; raw lines would flatten Oceania and hide the answer). A bar chart would drop the time dimension.

Colour: Grey for the five other continents and one strong blue (#0072B2) for Africa, the answer, drawn on top and thicker. Colour marks "the continent the question is about", not decoration. Lines are labelled directly at their ends instead of with a legend; the three almost identical lines (Asia, South America, Oceania) share one label so the text does not overlap.

In [6]:
# Q1
cont = df_long.groupby(['Continent', 'Year'], as_index=False)['Population'].sum()
base = cont[cont['Year'] == 1970].set_index('Continent')['Population']
cont['Index'] = cont['Population'] / cont['Continent'].map(base) * 100      # 1970 = 100
last = cont[cont['Year'] == 2022].set_index('Continent')['Index']

cmap = {c: GREY for c in cont['Continent'].unique()}
cmap['Africa'] = BLUE
order = [c for c in sorted(cmap) if c != 'Africa'] + ['Africa']              # Africa drawn last = on top

fig = px.line(cont, x='Year', y='Index', color='Continent', markers=True,
              color_discrete_map=cmap, category_orders={'Continent': order}, template='plotly_white')
fig.update_traces(line_width=2, marker_size=5)
fig.update_traces(line_width=4, marker_size=7, selector=dict(name='Africa'))

title = (f"Africa's population grew {last['Africa'] / 100:.1f}× since 1970, far faster than any other "
         f"continent;<br>Europe grew only {last['Europe'] - 100:.0f}%")
tidy(fig, title, r=260)

def end_label(y, text, color):
    fig.add_annotation(x=2022, y=y, text=text, showarrow=False, xanchor='left', xshift=8,
                       font=dict(color=color, size=12))

end_label(last['Africa'], f"<b>Africa ×{last['Africa'] / 100:.1f}</b>", BLUE)
end_label(last['North America'], f"North America ×{last['North America'] / 100:.1f}", '#777777')
cluster = last[['Asia', 'South America', 'Oceania']]
end_label(cluster.mean(), f"Asia, S. America, Oceania<br>×{cluster.min() / 100:.1f}–{cluster.max() / 100:.1f}", '#777777')
end_label(last['Europe'], f"Europe ×{last['Europe'] / 100:.1f}", '#777777')

fig.add_hline(y=100, line_dash='dot', line_color='#999999')
fig.update_xaxes(tickvals=[1970, 1980, 1990, 2000, 2010, 2020], range=[1968, 2023])
fig.update_yaxes(title='Population index (1970 = 100)')
fig.show()


---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

In [ ]:
summery:

The question asks for the 15 countries with the largest population increases.
This is a ranking problem involving one numerical variable: additional population.
Horizontal bars make it easy to compare the values.
Sorting the bars makes the ranking immediately understandable.
Horizontal orientation allows long country names to remain readable.
Highlighting African countries in blue makes it easy to count how many African countries appear in the top 15.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

Chart: Sorted horizontal bar chart of the 15 largest absolute gains. The task is a ranking of one number across categories, and bars compare length on a common zero baseline, which is the most accurate visual comparison. Bars are horizontal so the country names stay readable without rotation, and sorted by value because countries have no natural order. A table would work too, but it would not show the gaps (India's gain is more than 5× Nigeria's) at a glance.

Colour:Blue for Africa, grey for every other continent, so the second half of the question ("how many are in Africa?") can be answered by counting blue bars. Africa has the same blue as in Q1. Values are written on the bars, so no axis or legend is needed.

In [7]:
# Q2
df['Added'] = df['2022 Population'] - df['1970 Population']
top = df.nlargest(15, 'Added').copy()
top['Added_m'] = top['Added'] / 1e6                                         # millions of people

cmap = {c: GREY for c in df['Continent'].unique()}
cmap['Africa'] = BLUE

fig = px.bar(top, x='Added_m', y='Country', color='Continent', orientation='h',
             color_discrete_map=cmap, text='Added_m', template='plotly_white')
fig.update_traces(texttemplate='%{x:,.0f} M', textposition='outside', cliponaxis=False)

n_africa = int((top['Continent'] == 'Africa').sum())
india_china = top.set_index('Country').loc[['India', 'China'], 'Added'].sum() / 1e9
title = (f"{n_africa} of the 15 countries that added the most people since 1970 are in Africa,<br>"
         f"but India and China alone added {india_china:.2f} billion")
tidy(fig, title, height=560, r=80)

fig.update_yaxes(categoryorder='total ascending', gridcolor='white')      # largest on top
fig.update_xaxes(rangemode='tozero', showticklabels=False)                # zero baseline
fig.add_annotation(xref='paper', yref='paper', x=1, y=0.02, xanchor='right', showarrow=False,
                   text=f'<span style="color:{BLUE}">■</span> Africa   <span style="color:{GREY}">■</span> other continents')
fig.show()


---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

Chart:Scatter plot of land area against 2022 population, one point per country. Both variables are numeric and the question is about their relationship, so position on two axes is the right encoding. Both axes are logarithmic because area and population each span six to seven orders of magnitude (Vatican City to Russia / China); on linear axes nearly all countries would be squeezed into one corner. A dashed trend line gives the "expected" population for a given area, which makes the outliers visible as distance from the line. A bubble chart would need a third variable the question does not ask for.

Colour: One colour per continent (colour-blind-safe Okabe-Ito, the same colours as in Q4 and Q6) because the question asks whether the pattern differs by continent; each continent also has its own marker shape, so the chart does not rely on colour alone. The country labels are placed directly on the biggest outliers (far above or below the line) instead of in hover text.

In [8]:
# Q3
x, y = np.log10(df['Area']), np.log10(df['2022 Population'])
slope, intercept = np.polyfit(x, y, 1)
r = np.corrcoef(x, y)[0, 1]

fig = px.scatter(df, x='Area', y='2022 Population', color='Continent', symbol='Continent',
                 color_discrete_map=CONT_COLORS, log_x=True, log_y=True, hover_name='Country',
                 opacity=0.85, template='plotly_white')
fig.update_traces(marker_size=8)

fit_x = np.array([df['Area'].min(), df['Area'].max()])
fig.add_scatter(x=fit_x, y=10 ** (intercept + slope * np.log10(fit_x)), mode='lines',
                line=dict(color='#555555', dash='dash', width=1.5), showlegend=False, hoverinfo='skip')

title = (f"Bigger countries tend to have more people (r = {r:.2f} on log scales), but India, Bangladesh and<br>"
         "city-states sit far above the trend and Greenland, Mongolia and Libya far below")
tidy(fig, title, height=600)
fig.update_layout(showlegend=True, legend=dict(orientation='h', y=1.08, title_text=''))
fig.update_xaxes(title='Land area in km² (log scale)')
fig.update_yaxes(title='Population 2022 (log scale)')

def label(name, ax, ay):
    row = df[df['Country'] == name].iloc[0]
    # on a log axis, annotation coordinates are log10 values
    fig.add_annotation(x=np.log10(row['Area']), y=np.log10(row['2022 Population']), text=name,
                       showarrow=True, arrowhead=0, arrowwidth=0.8, arrowcolor='#999999',
                       ax=ax, ay=ay, font=dict(size=11, color='#333333'))

for name, ax, ay in [('India', -45, -15), ('China', 40, -25), ('Bangladesh', -60, -12),
                     ('Singapore', -55, -12), ('Hong Kong', 55, -22), ('Macau', 50, -18),
                     ('Greenland', 55, 10), ('Mongolia', -55, 22), ('Libya', 55, 5),
                     ('Canada', 45, -22), ('Australia', -55, 12)]:
    label(name, ax, ay)
fig.show()


---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

Chart:Violin plot (with an inner box and every country as a dot) per continent. The question asks for the *spread*, not only the average, so a chart of one value per continent (bar of means) would hide it; the violin shows the whole shape of the distribution, the inner box gives the median and quartiles, and the dots keep the small samples honest (14 to 57 countries per continent). The density axis is logarithmic (values run from 0.03 to 23,000 people/km²); on a linear axis everything except a few city-states would be a flat line at the bottom. Continents have no natural order, so they are sorted by median density.

Colour: The same continent colours as in Q3 and Q6 (one category = one colour). Colour only repeats the x-axis category, so no legend is needed; the median is written under each continent name. The most extreme countries are labelled directly.

In [9]:
# Q4
# Violin shapes are computed from the plotted values, so compute them on log10(density)
# (a KDE on raw values would be dominated by a few extreme city-states) and relabel the axis.
df['log_density'] = np.log10(df['Density'])
med = df.groupby('Continent')['Density'].median().sort_values(ascending=False)
order = med.index.tolist()

fig = px.violin(df, x='Continent', y='log_density', color='Continent', box=True, points='all',
                color_discrete_map=CONT_COLORS, category_orders={'Continent': order},
                hover_name='Country', hover_data={'Density': ':,.1f', 'log_density': False, 'Continent': False},
                template='plotly_white')
fig.update_traces(spanmode='hard', marker_size=4, meanline_visible=False)

n_asia = int(((df['Continent'] == 'Asia') & (df['Density'] > 5000)).sum())
n_eur = int(((df['Continent'] == 'Europe') & (df['Density'] > 5000)).sum())
title = (f"Typical density runs from ~{med['South America']:.0f} people per km² in South America to "
         f"~{med['North America']:.0f} in North America,<br>"
         f"while city-states push {n_asia} Asian and {n_eur} European countries above 5,000")
tidy(fig, title, height=600)
fig.update_xaxes(tickvals=order, ticktext=[f"{c}<br>median {med[c]:,.0f}" for c in order])
fig.update_yaxes(tickvals=[-1, 0, 1, 2, 3, 4], ticktext=['0.1', '1', '10', '100', '1,000', '10,000'],
                 title='People per km² (log scale)', range=[-1.8, 4.6])

for name, ax, ay in [('Macau', 60, 5), ('Monaco', 60, 5), ('Greenland', 60, -5)]:
    row = df[df['Country'] == name].iloc[0]
    fig.add_annotation(x=row['Continent'], y=row['log_density'], text=name, showarrow=True,
                       arrowhead=0, arrowcolor='#999999', ax=ax, ay=ay, font=dict(size=11, color='#333333'))
fig.show()


---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

Chart:Choropleth map of the percentage change in population from 2000 to 2022. The question has a spatial part ("where in the world are they?"), and a map is the only chart that answers it; the colour of each country directly encodes grow vs. shrink. A sorted bar chart of 234 countries would be unreadable and would lose location. The change is a *rate*, so I colour by % change, which is not additive and therefore correct for a map but not for a treemap.

Colour: A diverging scale with its midpoint fixed at 0 %: blue = growth, orange/vermillion = shrinkage, white = no change, a blue-orange pair that stays distinguishable under all common types of colour blindness (no red/green). The scale is capped at +100 % because a few countries (Qatar +317 %, UAE +188 %) would otherwise flatten all other differences; the cap and the number of capped countries are stated on the map. Here colour encodes a continuous value, not a continent, so it does not clash with the continent colours in other charts.

In [10]:
# Q5
cap = 100
df['Change'] = (df['2022 Population'] / df['2000 Population'] - 1) * 100
df['Change_capped'] = df['Change'].clip(upper=cap)
n_capped = int((df['Change'] > cap).sum())
n_grew, n_shrank = int((df['Change'] > 0).sum()), int((df['Change'] < 0).sum())
n_eu = int(((df['Change'] < 0) & (df['Continent'] == 'Europe')).sum())

lo = -40                                              # shrinkage in the data reaches about -38 %
zero_pos = (0 - lo) / (cap - lo)                      # white sits exactly at 0 %
scale = [[0, '#D55E00'], [zero_pos, '#FFFFFF'], [1, '#0072B2']]

fig = px.choropleth(df, locations='CCA3', locationmode='ISO-3', color='Change_capped',
                    hover_name='Country', hover_data={'Change': ':.1f', 'Change_capped': False, 'CCA3': False},
                    color_continuous_scale=scale, range_color=(lo, cap), template='plotly_white')
title = (f"Population grew in {n_grew} of {len(df)} countries and territories since 2000;<br>"
         f"the {n_shrank} that shrank are mostly in Europe ({n_eu})")
tidy(fig, title, height=560, r=20)
fig.update_geos(showframe=False, showcoastlines=False, showcountries=True, countrycolor='#DDDDDD',
                projection_type='natural earth', bgcolor='white')
fig.update_layout(coloraxis_colorbar=dict(
    title='Change 2000–2022', orientation='h', y=0.02, x=0.12, len=0.35, thickness=10,
    tickvals=[-40, -20, 0, 50, 100], ticktext=['−40%', '−20%', '0%', '+50%', '≥ +100%']))
fig.add_annotation(xref='paper', yref='paper', x=0, y=-0.02, xanchor='left', showarrow=False, align='left',
                   font=dict(size=11, color='#555555'),
                   text=f'Growth above {cap}% is shown at {cap}% (n = {n_capped} countries, e.g. Qatar +317%). '
                        'Source: UN World Population Prospects via Worldometer.')
fig.show()


---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

**Chart:** **Treemap with continent → country as the hierarchy. The question asks for the 2022 population at **two levels in one figure**; a treemap shows part-to-whole at both levels at once, with area proportional to population. It is appropriate because the value (people) is **additive**, so each continent's rectangle is the exact sum of its countries. A sorted bar chart could show only one level, and a pie chart cannot show the second level at all.

**Colour:** Each continent keeps the colour it has in Q3 and Q4 (colour-blind-safe Okabe-Ito), so colour identifies the level above and the country tiles inherit it. Labels are written on the tiles with the share of the world, so there is no legend; tiles too small for a label are left unlabelled (hover shows them), and the title states the key numbers.

In [11]:
# Q6
total = df['2022 Population'].sum()
asia = df.loc[df['Continent'] == 'Asia', '2022 Population'].sum() / total * 100
ci = df.loc[df['Country'].isin(['China', 'India']), '2022 Population'].sum() / total * 100

fig = px.treemap(df, path=['Continent', 'Country'], values='2022 Population',
                 color='Continent', color_discrete_map={**CONT_COLORS, '(?)': 'white'}, template='plotly_white')
fig.update_traces(texttemplate='%{label}<br>%{percentRoot:.1%}', textposition='middle center',
                  marker_line_width=1, marker_line_color='white', root_color='white',
                  hovertemplate='%{label}<br>%{value:,.0f} people<extra></extra>')
fig.update_layout(uniformtext=dict(minsize=9, mode='hide'))

title = (f"Asia is home to {asia:.0f}% of the world's {total / 1e9:.1f} billion people; "
         f"China and India alone are {ci:.0f}%")
tidy(fig, title, height=620, r=20)
fig.update_layout(margin=dict(t=100, l=10, r=10, b=40))
fig.add_annotation(xref='paper', yref='paper', x=0, y=-0.05, xanchor='left', showarrow=False,
                   font=dict(size=11, color='#555555'),
                   text='Area = 2022 population; percentages are shares of the world total. Source: UN WPP via Worldometer.')
fig.show()


---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.
Chart:100 % stacked bar chart, one bar per year. The question is about *shares of a whole* changing over time, and 100 % stacking removes the growth of the total (which is not the question). The years have a natural order, so they stay in chronological order on the x-axis. Africa is the first segment, sitting on the baseline, so its height can be compared across years directly; segments that float on top of other segments cannot be compared that accurately. A line chart would also work, but Q1 already uses one, and the question is about composition of the whole.

Colour: One accent, blue for Africa, the share that is rising; Asia, Europe and "Rest of the world" are three light shades of grey so they remain distinguishable but recede. The lightness contrast still works in greyscale. Segments and series names are labelled directly on the chart, so no legend is needed. The footnote notes that the years are not evenly spaced.

In [14]:
import plotly.io as pio
pio.renderers.default = 'notebook'

In [18]:

# Q7 - Share of world population by continent

import numpy as np
import plotly.express as px

# Group continents
df_long['Group'] = np.where(
    df_long['Continent'].isin(['Africa', 'Asia', 'Europe']),
    df_long['Continent'],
    'Rest of the world'
)

# Calculate population by year and group
sh = df_long.groupby(
    ['Year', 'Group'], as_index=False
)['Population'].sum()

# Calculate percentage shares
sh['Share'] = (
    sh['Population'] /
    sh.groupby('Year')['Population'].transform('sum') * 100
)

# Ensure chronological order
sh['Year'] = sh['Year'].astype(str)

year_order = sorted(sh['Year'].unique(), key=int)

# Region order and colours
order = ['Africa', 'Asia', 'Europe', 'Rest of the world']

cmap = {
    'Africa': BLUE,
    'Asia': '#9A9A9A',
    'Europe': '#BDBDBD',
    'Rest of the world': '#DDDDDD'
}

# Create stacked bar chart
fig = px.bar(
    sh,
    x='Year',
    y='Share',
    color='Group',
    color_discrete_map=cmap,
    category_orders={
        'Group': order,
        'Year': year_order
    },
    barmode='stack',
    template='plotly_white'
)

# Add percentage labels
fig.update_traces(
    texttemplate='%{y:.1f}%',
    textposition='inside',
    insidetextanchor='middle',
    textfont=dict(size=10, color='#333333')
)

# Highlight Africa labels in white
fig.update_traces(
    textfont_color='white',
    selector=dict(name='Africa')
)

# Calculate title values
p = sh.pivot(
    index='Year',
    columns='Group',
    values='Share'
)

title = (
    f"Africa's share of world population rose from "
    f"{p.loc['1970', 'Africa']:.1f}% to "
    f"{p.loc['2022', 'Africa']:.1f}%, "
    f"while Europe's fell from "
    f"{p.loc['1970', 'Europe']:.1f}% to "
    f"{p.loc['2022', 'Europe']:.1f}%; "
    f"Asia held steady at "
    f"{p['Asia'].min():.0f}–{p['Asia'].max():.0f}%"
)

# Chart layout
fig.update_layout(
    title=dict(
        text=title,
        x=0,
        xanchor='left',
        font=dict(size=17, color='#17365D')
    ),
    height=620,
    width=1100,
    barmode='stack',
    bargap=0.25,

    xaxis=dict(
        title='Year',
        categoryorder='array',
        categoryarray=year_order
    ),

    yaxis=dict(
        title='Share of total world population',
        range=[0, 100],
        ticksuffix='%',
        dtick=20
    ),

    legend=dict(
        title='',
        orientation='h',
        y=1.04,
        x=0,
        xanchor='left',
        yanchor='bottom'
    ),

    margin=dict(
        l=70,
        r=40,
        t=145,
        b=115
    )
)

# Add ONE source note only
fig.add_annotation(
    xref='paper',
    yref='paper',
    x=0,
    y=-0.19,
    xanchor='left',
    yanchor='top',
    showarrow=False,
    text=(
        'Shares of total world population. '
        'Source: UN WPP via Worldometer.'
    ),
    font=dict(size=11, color='#666666')
)

# Display chart
fig.show()
